In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [ ]:
colab = 1
if colab ==True:
  data_dir='/content/drive/My Drive/Colab Notebooks/big_data_codes/SAheart.csv'
  from google.colab import drive
  drive.mount('/content/drive')
else:
  data_dir='SAheart.csv'


# 2-1

## data processing and plot

### import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
# sns.set(style="whitegrid")
import warnings
from sklearn.svm import SVC, NuSVC
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors  import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from scipy import stats
from scipy.stats import uniform, randint
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import roc_curve, auc, accuracy_score
# from tflearn.data_utils import to_categorical
from sklearn import preprocessing
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
from scipy import interp
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
from sklearn.decomposition import FastICA
from keras.utils import to_categorical
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import VotingClassifier


### make the dataset file SAheart.csv

In [ ]:
import pandas as pd
import requests
from io import StringIO
import os

# Define the URL
url = "https://hastie.su.domains/ElemStatLearn/datasets/SAheart.data"

# Specify the path in your Google Drive
# save_path = '/content/drive/My Drive/Colab Notebooks/big_data_codes/SAheart.csv'
save_path = '/content/drive/My Drive/Colab Notebooks/big_data_codes/SAheart.csv'

# Check if the file already exists
if os.path.exists(save_path):
    # If it exists, read it as a DataFrame
    df = pd.read_csv(save_path)
    # Display the first five rows
    print("DataFrame from existing SAheart.csv:\n")
    df.head()
else:
    # If it doesn't exist, download the data, create DataFrame, and save it
    response = requests.get(url)
    data = response.text

    # Read the data into a DataFrame
    columns = [
        "row",
        "sbp",
        "tobacco",
        "ldl",
        "adiposity",
        "famhist",
        "typea",
        "obesity",
        "alcohol",
        "age",
        "chd",
    ]
    df = pd.read_csv(StringIO(data), sep=",", header=None, names=columns, na_values="NA")

    # Drop the first row
    df = df.iloc[1:]

    # Drop the first column
    df = df.iloc[:, 1:]

    encoder = OrdinalEncoder()

    # Reshape the 'famhist' column to a 2D array
    df['famhist'] = encoder.fit_transform(df['famhist'].values.reshape(-1, 1))

    # Save the DataFrame to a CSV file in Google Drive
    df.to_csv(save_path, index=False)

    # Display the DataFrame
    print("New DataFrame created and saved as SAheart.csv:\n")
    df.head()


### Utility Functions

In [ ]:
Renamed_feature= []               #list of names that will rename to feature column
all_clf_res=[]                    #every classifier auc values are stored in it
random_initializer=100            #random initializer
n_dots=50
##########################################################

for i in range(9):
  #for renaming dataset of columns features F1 -- F9
  Renamed_feature.append('F'+str(i+1))
############################################################

# Pairs plots are just showing all variables paired with all the other variables
def pair_plot(data):
  '''
  This function will create a grid of Axes such that each variable
  in data will by shared in the y-axis across a single row and in the x-axis
  across a single column.The diagonal Axes are treated differently, drawing
  a plot to show the univariate distribution of the data for the variable in
  that column.
  Parameters :
  Input - data is the pandas type variable for
  plotting pair plot of features in this
  dataframe

  Output :
  This function Plot pairwise relationships in a dataset.
  '''
  plt.figure()

  pair_plot =sns.pairplot(data=data,
                          height=3,
                          hue='chd',
                          diag_kind='kde')
  pair_plot.fig.suptitle("Pairplot of all features")
  plt.show()




###################################################################
# this function for Gaussian distribution plot
# and box plot simultaneously in a figure
def Box_Gaussian(data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output - The Gaussian distribution plot for
  eight feature of input data
  '''

  fig=plt.figure(figsize=(15,6))                                #define figure size
  fig.suptitle("Box Gaussian plot of all features")

  n_scaler = preprocessing.StandardScaler()                 #standardization function
  temp_Data = n_scaler.fit_transform(data)                  #pass into function for standrd.
  for i in range(9):                                        #loop for all 9 feature

    plt.subplot(3, 3, i+1)                                  #subplot for 3 rows in 3 columns
    Data = temp_Data[:,i]                                   #data for every feature
    sns.kdeplot(Data, shade=True,color='red', alpha=0.3)    #kernel density function under red shaded arae
    ax = sns.boxplot(Data, saturation=0.9, color="green")   #boxplot  with green shaded area
                                                            # https://seaborn.pydata.org/generated/seaborn.kdeplot.html
                                                            # https://seaborn.pydata.org/generated/seaborn.boxplot.html
    plt.gca().invert_yaxis()                                #Reverse Y-Axis in PyPlot
    # plt.title('F'+str(i+1))
    frame1 = plt.gca()
    frame1.axes.xaxis.set_ticklabels([])                    #removing xlabel data
    plt.ylim((-0.5,0.65))                                   #y axis  limit
    plt.tight_layout()                                      #This module provides routines to adjust subplot params so that subplots are nicely fit in the figure.
                                                            # https://matplotlib.org/api/tight_layout_api.html
    # plt.grid('on')

    for patch in ax.artists:
      r, g, b, a = patch.get_facecolor()                     #Get the facecolor of the Axes.
      patch.set_facecolor((r, g, b, 0.3))                    #set colour intensity
#############################################################


def plot_confusionMatrix(data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output -visualization of correalation matrix of
  input data
  '''
  sns.set(font_scale=1.15)                                    # Set aesthetic parameters in one step.
  ax = plt.figure(figsize=(10, 8))                            #set figure size   https://seaborn.pydata.org/generated/seaborn.set.html
  plt.title("Confusion Matrix of all features")
  sns.heatmap(data.corr(),                                    # input correlation matrix  of dataset
              vmax=1.0,                                       #Values to anchor the colormap, otherwise they are inferred from
                                                              #the data and other keyword arguments.
              vmin=0.0,
              linewidths=0.01,
              square=False,                                   #If True, set the Axes aspect to “equal” so each cell will be square-shaped.
              annot=True,                                     #If True, write the data value in each cell.
              linecolor="black")                              #Color of the lines that will divide each cell.
                                                              #https://seaborn.pydata.org/generated/seaborn.heatmap.html
  b, t = plt.ylim()                                           # discover the values for bottom and top
  b += 0.5                                                    # Add 0.5 to the bottom
  t -= 0.5                                                    # Subtract 0.5 from the top
  plt.ylim(b, t)                                              # update the ylim(bottom, top) values
  plt.show()



############################################################
# this function plot univariate distribution of
# every feature

def dist_Plot(data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output - The distribution plot for
  eight feature of input data
  '''
  fig, ax = plt.subplots(3,3, figsize=(12,5))                 #set numbers of rows and columns of subplot
  sns.set()
  sns.distplot(data.F1, bins = 10, ax=ax[0,0])                #Flexibly plot a univariate distribution of observations.
  sns.distplot(data.F2, bins = 10, ax=ax[0,1])
  sns.distplot(data.F3, bins = 10, ax=ax[0,2])
  sns.distplot(data.F4, bins = 10, ax=ax[1,0])
  sns.distplot(data.F5, bins = 10, ax=ax[1,1])
  sns.distplot(data.F6, bins = 10, ax=ax[1,2])
  sns.distplot(data.F7, bins = 10, ax=ax[2,0])
  sns.distplot(data.F8, bins = 10, ax=ax[2,1])
  sns.distplot(data.F9, bins = 10, ax=ax[2,2])
  fig.suptitle("Gaussian Distribution of all features")
  fig.tight_layout()                                          #This module provides routines to adjust subplot params
                                                              #  so that subplots are nicely fit in the figure.


############################################################
# this function plot violin plot  of
# every feature


def plot_violinplot (data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output - The violinplot plot for
  eight feature of input data
  '''

  # A violin plot is a method of plotting numeric data.
  # It is similar to box plot with a rotated kernel
  # density plot on each side. Violin plots are similar
  # to box plots, except that they also show the probability
  # density of the data at different values (in the simplest
  # case this could be a histogram).
  fig, ax = plt.subplots(3,3, figsize=(12,6))
  # #set numbers of rows and columns of subplot and figure size
  sns.set()
  sns.violinplot(x = data.chd, y=data.F1,  ax=ax[0,0])    #violine plot for F1 feature
  sns.violinplot(x = data.chd, y=data.F2,  ax=ax[0,1])    #violine plot for F2 feature
  sns.violinplot(x = data.chd, y=data.F3,  ax=ax[0,2])    #violine plot for F3 feature
  sns.violinplot(x = data.chd, y=data.F4,  ax=ax[1,0])    #violine plot for F4 feature
  sns.violinplot(x = data.chd, y=data.F5,  ax=ax[1,1])    #violine plot for F5 feature
  sns.violinplot(x = data.chd, y=data.F6,  ax=ax[1,2])    #violine plot for F6 feature
  sns.violinplot(x = data.chd, y=data.F7,  ax=ax[2,0])    #violine plot for F7 feature
  sns.violinplot(x = data.chd, y=data.F8,  ax=ax[2,1])    #violine plot for F8 feature
  sns.violinplot(x = data.chd, y=data.F9,  ax=ax[2,2])    #violine plot for F9 feature
  fig.suptitle("Violin plot of all features")
  fig.tight_layout()

                                                              # https://seaborn.pydata.org/generated/seaborn.violinplot.html

############################################################

# this function if for outlair rejection with
# respect to mean value
def IQR_Mean (data):

  '''
  Parameters :
  Input - data is the pandas type variable

  Return - dataframe with outleir rejection
  filled with mean
  of input data
  '''
  for i in range(9):
    x = data[Renamed_feature[i]]
    Q1 = x.quantile(0.25)                                   # Q1 is the "middle" value in the first half of the rank-ordered data set.
    Q3 = x.quantile(0.75)                                   # Q3 is the "middle" value in the second half of the rank-ordered data set.
    IQR = Q3-Q1                                             # The interquartile range is equal to Q3 minus Q1.
    mean = x.mean()                                         #mean of feature
    for j in range(462):                                    # loop for first 462 elements of feature
      temp = x[j]                                           # every feature value
      LW = (Q1 - 1.5 * IQR)                                 #lower considerable range of gaussian distribution
      UW = (Q3 + 1.5 * IQR)                                 #upper considerable range of gaussian distribution
      if temp < LW:                                         #replace upper value with mean
        x[j] = mean
      if temp > UW:                                         #replace lower value with mean
        x[j] = mean
    data[Renamed_feature[i]] = x
  return data

############################################################
# this function if for outlair rejection with
# respect to median value same as previous function
def IQR_Medain (data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Return - dataframe with outleir rejection
  filled with median of input data
  '''
  for i in range(9):
    x = data[Renamed_feature[i]]
    Q1 = x.quantile(0.25)
    Q3 = x.quantile(0.75)
    IQR = Q3-Q1
    median = x.quantile(0.5)                                # find the median
    for j in range(462):                                    #replace the first 462 values with respect to median
      temp = x[j]
      LW = (Q1 - 1.5 * IQR)
      UW = (Q3 + 1.5 * IQR)
      if temp < LW:                                         #replace upper value with median
        x[j] = median
      if temp > UW:
        x[j] = median                                       #replace upper value with median
    data[Renamed_feature[i]] = x
  return data

############################################################



# this function if for outlair rejection with
# 1.5 times of IQR that means that are
# significant in gaussian distribution
def IQR (data):

  '''
  Parameters :
  Input - data is the pandas type variable

  Return - dataframe with outleir rejection
  of input data

  '''
  #input dataset as data
  for i in range(9):                                        # for every feature
    Q1 = data[Renamed_feature[i]].quantile(0.25)
    Q3 = data[Renamed_feature[i]].quantile(0.75)
    IQR = Q3-Q1                                             #find IQR
    LW = (Q1 - 1.5 * IQR)                                   #find lower boundary
          # print(LW)
    UW = (Q3 + 1.5 * IQR)                                   #find upper boundary
          # print(UW)
    data = data[data[Renamed_feature[i]]<UW]                #drop greater than upper limit
    data = data[data[Renamed_feature[i]]>LW]                #drop smaller than lower limit

  return data


############################################################
#outlier rejection with different condition

def outlier_Rejection (data, iqr_Mean, iqr_Medain, iqr):
  '''
  Parameters :
  Input -
  data is the pandas type variable
  iqr_Mean - for outleir rejection with Mean
  iqr_Medain- for outleir rejection with Medain
  iqr- for drop the outleir
  manual -for manual rejection
  Return - dataframe with outleir rejection
  filled with Input parameter

  '''

  # outlier_Rejection with conditional input
  if iqr_Mean == True:                                     #reject outleir with Mean
    data = IQR_Mean (data)
  if iqr_Medain == True:                                   #reject outleir with Median
    data = IQR_Medain (data)
  if iqr == True:                                          #reject outleir in IQR range
    data = IQR (data)

  return data

############################################################

#data plot on different input condition
def data_plot (data,
               Pair_plot,
               Dist_Plot,
               Plot_violinplot,
               Plot_confusionMatrix,
               box_Gaussian):

  '''
  Parameters :
  Input -
  data - It is the pandas type variable
  Pair_plot - for pair plot visualization of input  data
  Dist_Plot- for gaussian distribution plot visualization of input  data
  Plot_violinplot- for violin plot visualization of input  data
  Plot_confusionMatrix -for confusion matrix visualization of input  data

  Output - dataframe with outleir rejection
  filled with Input parameter

  '''
  if Pair_plot ==True:
    pair_plot(data)

  if Dist_Plot ==True:
    dist_Plot(data)

  if Plot_violinplot ==True:
    plot_violinplot (data)

  if Plot_confusionMatrix ==True:
    plot_confusionMatrix(data)

  if box_Gaussian ==True:
    Box_Gaussian(data)


############################################################


def metrics (y_true, y_pred, probas_):


  '''
  Parameters :
  Input -
  y_true - true  value of input data
  y_pred- predicted  value of input data
  probas_- probability/confidence of predicted output

  return -True Negative(tn),False Positive(fp),False Negative(fn)
  True positive(tp),AUC(roc_auc),False Positive Rate(fpr),
  True positive rate(tpr)

  '''


  points=n_dots*'-'
  print(points)
#    print("Best parameters set found on development set:")
#    print(clf.best_params_)
  fpr, tpr, thresholds = roc_curve(y_true, probas_[:, 1])
  tprs.append(interp(mean_fpr, fpr, tpr))
  tprs[-1][0] = 0.0
  roc_auc = auc(fpr, tpr)
  #  aucs.append(roc_auc)
  print("Detailed classification report for current fold:")
  print()
  print(classification_report(y_true, y_pred))
  print()
  print("Area Under ROC (AUC): {}".format(roc_auc))
  print()
  print('Confusion Matrix for current fold: ')
  print(confusion_matrix(y_true, y_pred))
  print()
  print("Accuracy for Current Fold: {}".format(accuracy_score(y_true, y_pred)))
  print()
  tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

  return  tn, fp, fn, tp, roc_auc, fpr, tpr

############################################################


def average_ROC(mean_fpr,tprs,aucs,TP,TN,FP,FN):

  '''
  Parameters :
  mean_fpr - Mean False positive rate
  tprs -values of true positive rate
  aucs  - values of auc
  TP    - True positive
  TN    - True Negative
  FP    - False Positiv
  FN    - False Negative

  Output -
  Visalization of TPR vs FPR plot
  '''
  sen = (np.sum(TP))/(np.sum(TP)+np.sum(FN))
  spe = (np.sum(TN))/(np.sum(TN)+np.sum(FP))

  mean_tpr = np.mean(tprs, axis=0)
  mean_tpr[-1] = 1.0
  # mean_auc = auc(mean_fpr, mean_tpr)
  mean_auc = np.mean(aucs)
  std_auc = np.std(aucs)
  # plt.figure(figsize=(8, 5))
  # plt.grid(True)
  ax = plt.axes()
  ax.grid(color='lightgray', linestyle='-', linewidth=.5)
  # Setting the background color
  ax.set_facecolor("white")

  ax.spines['bottom'].set_color('#000000')
  ax.spines['top'].set_color('#000000')
  ax.spines['right'].set_color('#000000')
  ax.spines['left'].set_color('#000000')

  plt.plot(mean_fpr, mean_tpr, color='blue',
          label=r'Avg. ROC (AUC (avg $\pm$ std) = %0.3f $\pm$ %0.3f)' % (mean_auc, std_auc),
          lw=2, alpha=.8)

  plt.scatter((1-spe), sen, s=80, c='r', marker='x',)
  plt.scatter(0, sen, s=80, c='r', marker='x',)
  plt.scatter((1-spe),0, s=80, c='r', marker='x',)
  plt.axhline(y=sen, color='r', linestyle='--')
  plt.axvline(x=(1-spe), color='r', linestyle='--')
  plt.text((1-spe), 0.02, "FPR={:0.3f}".format((1-spe)))
  plt.text(0.009, sen+0.05, "TPR={:0.3f}".format(sen))

  std_tpr = np.std(tprs, axis=0)
  tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
  tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
  plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='darkgray', alpha=0.5,
                  label=r'$\pm$ 1 Standard deviation')

  plt.xticks(np.arange(0.0, 1.01, step=0.1))
  plt.yticks(np.arange(0.0, 1.01, step=0.1))
  left=0.0
  right=1.0
  plt.xlim(left, right)
  plt.ylim(left, right)
  plt.xlabel('False Positive Rate (FPR)')
  plt.ylabel('True Positive Rate (TPR)')
  plt.legend(loc="lower right")
  # plt.grid(True)
  plt.show()

############################################################

def plot_Current_ROC(fpr,tpr,iterator,roc_auc):

  '''
  Parameters :
  Input -
  fpr - False positive rate
  tpr - True positive rate
  roc_auc -auc values of roc curve

  Output -
  Visalization of current roc curve

  '''
  plt.plot(fpr,

          tpr,
          # Color[iterator],
          alpha=0.35,
          # label='macro-average ROC (AUC = {0:0.3f})'.format(roc_auc)
          # +FOLD[iterator],
          linewidth=1)

############################################################


def creat_Model (classifier, X_Train, Y_Train, tuned_parameters, verbose):

  '''
  Parameters :
  Input -
  X_Train -train data
  Y_Train - label/output of train data
  tuned_parameters =parameters of models
  verbose = condition about model

  Output -
  Returned a tuned classifier using grid search
  '''
  clf = GridSearchCV(classifier,
                    tuned_parameters,
                    verbose=verbose,
                    cv=5,
                    scoring='roc_auc',
                    n_jobs=None)
  clf.fit(X_Train, Y_Train)
  return clf
############################################################

def average_performance(aucs,Accuracy,TP,TN,FP,FN):

  '''
  Parameters :
  Input -
  aucs= values of aucs
  Accuracy - value of accuracy
  TP  - True Positive
  TN  - True Negative
  FP  - False Positive
  FN  - False Negative


  Output -
  It prints the average aucs,accuarcy,confusion matrix
  '''

  print()
  n_dotsav=(n_dots-len('Average'))//2

  print('-'*n_dotsav+'Average'+'-'*n_dotsav)
  print("AUC (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(aucs),np.std(aucs)))
  print("Accuracy (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(Accuracy),np.std(Accuracy)))
  cm = [[int(np.mean(TP)), int(np.mean(FP))],[int(np.mean(FN)), int(np.mean(TN))]]
  print ('Avg. CM is '+str(cm))
  cm = [[int(np.sum(TP)), int(np.sum(FP))],[int(np.sum(FN)), int(np.sum(TN))]]
  print ('Total for all folds CM is '+str(cm))
  re_auc=str(round(np.mean(aucs), 3))+'+/-'+str(round(np.std(aucs),3))
  all_clf_res.append(re_auc)

############################################################

#this  function is for algorithm based feature selection
def feature_Selector(data, algo, n_feature):
    '''
    Parameters :
    Input -
    data - It is the pandas type variable
    algo - type of algorith PCA,ICA,Correlation


    Output -
    It prints the average aucs,accuarcy,confusion matrix
    '''
    if algo=='PCA':                                                   #for pca algorithm
        X_Data= data.iloc[:,:9].values
        pca = PCA(n_components=n_feature)                             #number of feature
        X_Data = pca.fit_transform(X_Data)
        return X_Data , data.iloc[:,9:].values

    if algo == 'ICA':
        X_Data= data.iloc[:,:9].values
        ICA = FastICA(n_components=n_feature, random_state=12)
        X_Data = ICA.fit_transform(X_Data)
        return X_Data , data.iloc[:,9:].values

    if algo =='corr':                                                   #for ica algorithm
        if n_feature ==4:
            data = data[['F2','F5','F4','F6','Outcome']]                #for 4 feature
            return data.iloc[:,:4].values, data.iloc[:,4:].values
        if n_feature ==6:
            data = data[['F1','F2','F4','F5','F6','F8','Outcome']]       #for 6 feature
            return data.iloc[:,:6].values, data.iloc[:,6:].values

    if algo == 'None':
        return data.iloc[:,:9].values, data.iloc[:,9:].values            #if feature selection is off all features are counted


### Read the data from the drive using pandas (Python Data Analysis Library)


In [ ]:
if colab ==True:
  data = pd.read_csv(data_dir)
else:
  data = pd.read_csv(data_dir)
data.shape


In [ ]:
data.head(n=6)


### Renaming the Features by F1, F2, and so on ......
---
New Name | Original Name | Comments|
---|---|---|
F1|sbp|systolic blood pressure|
F2|tobacco|cumulative tobacco (kg)|
F3|ldl|low densiity lipoprotein cholesterol|
F4|adiposity|fatness |
F5|famhist|family history of heart disease (Present = 1, Absent = 0)|
F6|typea|type-A behavior|
F7|obesity|fatness |
F8|alcohol|current alcohol consumption|
F9|age|age at onset|


In [ ]:
data = pd.DataFrame({'F1':data.iloc[:,:9].values[:,0],
                     'F2':data.iloc[:,:9].values[:,1],
                     'F3':data.iloc[:,:9].values[:,2],
                     'F4':data.iloc[:,:9].values[:,3],
                     'F5':data.iloc[:,:9].values[:,4],
                     'F6':data.iloc[:,:9].values[:,5],
                     'F7':data.iloc[:,:9].values[:,6],
                     'F8':data.iloc[:,:9].values[:,7],
                     'F9':data.iloc[:,:9].values[:,8],
                     'chd':data.iloc[:,9:].values[:,0]})


In [ ]:
data.describe()


### Raw Data Plot and Presenation

In [ ]:
data_plot (data,
          Pair_plot=True,
          Dist_Plot=True,
          Plot_violinplot=True,
          Plot_confusionMatrix=True,
          box_Gaussian=False)


### Data Preprocessing  

In [ ]:
print('Shape Before Process: ' + str(data.shape))
##########################################################################

## The process for the outlier rejection (P)

data = outlier_Rejection (data,
                  iqr_Mean=False,
                  iqr_Medain=False,
                  iqr=True)
print('Shape After outlier Removed: ' + str(data.shape))

##########################################################################
#  algo parameters are
# 'PCA','ICA','corr','None'

X_Data,Y_Lavel = feature_Selector(data, algo='PCA', n_feature=5)
print('Shape After Feature Selection: ' + str(data.shape))


##########################################################################
# The process of Standardization  (S)
scaler =  preprocessing.StandardScaler()
X_Data,Y_Lavel= scaler.fit_transform(X_Data), Y_Lavel
print('Shape After Standardization: ' + str(data.shape))


##########################################################################
# Stratified K-Folds cross-validator
# Provides train/test indices to split
# data in train/test sets.This cross-validation
#  object is a variation of KFold that returns
#  stratified folds. The folds are made by preserving
#  the percentage of samples for each class.

kf = StratifiedKFold(n_splits=5,
                     shuffle=False)


In [ ]:
print(X_Data.shape)
print(Y_Lavel.shape)


### Processed Data Plot and Presenation

In [ ]:
data_plot (data,
          Pair_plot=True,
          Dist_Plot=True,
          Plot_violinplot=True,
          Plot_confusionMatrix=True,
          box_Gaussian=False)



## AdaBoost

In [ ]:
Accuracy = []                                                                # for store the value of accuracy
FP = []                                                                      # for store False Positive
TN = []                                                                      # for True Negative
FN = []                                                                      # for False Negative
TP = []                                                                      # for True Positive
tprs = []                                                                    # for true positive rates
aucs_aBoost = []                                                             # for store the values of auc of Adaboost model
iterator=0

mean_fpr = np.linspace(0, 1, 100)
fig = plt.figure(figsize=(8, 5))


for train_index, test_index in kf.split(X_Data,Y_Lavel):                     #split into train and test
    #   print("TRAIN:", train_index, "TEST:", test_index)
    X_Train, X_Test = X_Data[train_index], X_Data[test_index]                #data and label
    Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]              #data and label

    #####################################################
    # define the parameters of adaboost algorithm
    #####################################################
    tuned_parameters = { 'algorithm': ['SAMME','SAMME.R'],
                       'learning_rate':[0.1,0.5,1.0],
                       'n_estimators': [10,50,100,200]}


    clf = creat_Model (classifier = AdaBoostClassifier( random_state=random_initializer),
                      X_Train = X_Train,                                      # create a model using  AdaBoost Classifier
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters,
                      verbose=0)
    tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,             # model evaluation parametrs
                                                y_pred = clf.predict(X_Test),
                                                probas_ = clf.predict_proba(X_Test))
    tprs.append(interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    aucs_aBoost.append(roc_auc)
    plot_Current_ROC (fpr,tpr,iterator,roc_auc)                              #plot the roc of current fold
    iterator += 1
    TN.append(tn)
    FP.append(fp)
    FN.append(fn)
    TP.append(tp)
    Accuracy.append(accuracy_score(Y_Test, clf.predict(X_Test)))
average_ROC(mean_fpr,tprs,aucs_aBoost,TP,TN,FP,FN)                           #plot average roc curve
average_performance(aucs_aBoost,Accuracy,TP,TN,FP,FN)                        #print the average performance of the model


## KNN

In [ ]:
Accuracy = []                                                                # for store the value of accuracy
FP = []                                                                      # for store False Positive
TN = []                                                                      # for True Negative
FN = []                                                                      # for False Negative
TP = []                                                                      # for True Positive
tprs = []                                                                    # for true positive rates
aucs_kNN = []                                                                # for store the values of auc
iterator=0
mean_fpr = np.linspace(0, 1, 100)
fig = plt.figure(figsize=(8, 5))

for train_index, test_index in kf.split(X_Data,Y_Lavel):                     # split in train and test
    #   print("TRAIN:", train_index, "TEST:", test_index)
    X_Train, X_Test = X_Data[train_index], X_Data[test_index]                #train data and label
    Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]              #test data and label

    ###########################################
    # define the hyper parameters of Knn
    n_neighbors = [1,3,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,37,39,41,43,45,47,49]
    leaf_size = [5,10,15,20,25,30,35,40,45,50]
    Distance = [1,2]
    ############################################


    tuned_parameters = [ {'n_neighbors': n_neighbors,                         #define parameters with different algorithm
                        'algorithm' : ['brute'],
                        'p':Distance},

                         {'n_neighbors': n_neighbors,
                        'algorithm' : ['ball_tree'],
                        'leaf_size' : leaf_size,
                        'p':Distance},

                        {'n_neighbors': n_neighbors,
                        'algorithm' : ['kd_tree'],
                        'leaf_size' : leaf_size,
                        'p':Distance}]

    clf = creat_Model (classifier = KNeighborsClassifier(),                     #create the model
                      X_Train = X_Train,
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters,
                      verbose=0)

    tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,               #get the values of  model evaluation
                                                y_pred = clf.predict(X_Test),
                                                probas_ = clf.predict_proba(X_Test))
    tprs.append(interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    aucs_kNN.append(roc_auc)
    plot_Current_ROC (fpr,tpr,iterator,roc_auc)                                 #plot the roc of current fold
    iterator += 1
    TN.append(tn)
    FP.append(fp)
    FN.append(fn)
    TP.append(tp)
    Accuracy.append(accuracy_score(Y_Test, clf.predict(X_Test)))
average_ROC(mean_fpr,tprs,aucs_kNN,TP,TN,FP,FN)                                 #plot average roc curve
average_performance(aucs_kNN,Accuracy,TP,TN,FP,FN)                              #print the average performance of the model


## Random Forest

In [ ]:
Accuracy = []                                                                # for store the value of accuracy
FP = []                                                                      # for store False Positive
TN = []                                                                      # for True Negative
FN = []                                                                      # for False Negative
TP = []                                                                      # for True Positive
tprs = []                                                                    # for true positive rates
aucs_Forest = []                                                             # for store the values of auc of Random Forest model
iterator=0
mean_fpr = np.linspace(0, 1, 100)
fig = plt.figure(figsize=(8, 5))

for train_index, test_index in kf.split(X_Data,Y_Lavel):                     #split dataset into train /test
#   print("TRAIN:", train_index, "TEST:", test_index)
    X_Train, X_Test = X_Data[train_index], X_Data[test_index]                # data and label of train dataset
    Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]              # data and label of test dataset

    # parameters of Random Forest model
    tuned_parameters = {'criterion': ['gini','entropy']}

    clf = creat_Model (classifier = RandomForestClassifier( random_state=random_initializer),
                      X_Train = X_Train,                                      # create a model using random forest classifier
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters,
                      verbose=0)

    tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,              #evaluation parameters of random forest model
                                                y_pred = clf.predict(X_Test),
                                                probas_ = clf.predict_proba(X_Test))
    tprs.append(interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    aucs_Forest.append(roc_auc)
    plot_Current_ROC (fpr,tpr,iterator,roc_auc)                               #plot the roc of current fold
    iterator += 1
    TN.append(tn)
    FP.append(fp)
    FN.append(fn)
    TP.append(tp)
    Accuracy.append(accuracy_score(Y_Test, clf.predict(X_Test)))
average_ROC(mean_fpr,tprs,aucs_Forest,TP,TN,FP,FN)                            #plot average roc curve
average_performance(aucs_Forest,Accuracy,TP,TN,FP,FN)                         #print the average performance of the model

## Naive Bayes

In [ ]:
Accuracy = []                                                                # for store the value of accuracy
FP = []                                                                      # for store False Positive
TN = []                                                                      # for True Negative
FN = []                                                                      # for False Negative
TP = []                                                                      # for True Positive
tprs = []                                                                    # for true positive rates
aucs_NB = []                                                                 # for store the values of auc of  model
iterator=0

mean_fpr = np.linspace(0, 1, 100)
fig = plt.figure(figsize=(8, 5))


for train_index, test_index in kf.split(X_Data,Y_Lavel):                     #split into train and test
#   print("TRAIN:", train_index, "TEST:", test_index)
    X_Train, X_Test = X_Data[train_index], X_Data[test_index]                #train data and label
    Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]              #test  data and label


    #############################################
    # define parameters of Naive Bayes model
    ############################################
    var_smoothing = [1e-01,
                    1e-02,
                    1e-03,
                    1e-04,
                    1e-05,
                    1e-06,
                    1e-07,
                    1e-08,
                    1e-09,
                    1e-10,
                    1e-11,
                    1e-12]

    tuned_parameters = [{'var_smoothing': var_smoothing}]

    #############################################################
    clf = creat_Model (classifier = GaussianNB(),                             # create model using Naive Bias
                      X_Train = X_Train,
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters,
                      verbose=0)
    tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,             # model evaluation parameters
                                                y_pred = clf.predict(X_Test),
                                                probas_ = clf.predict_proba(X_Test))
    tprs.append(interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    aucs_NB.append(roc_auc)
    plot_Current_ROC (fpr,tpr,iterator,roc_auc)                               #plot the roc of current fold
    iterator += 1
    TN.append(tn)
    FP.append(fp)
    FN.append(fn)
    TP.append(tp)
    Accuracy.append(accuracy_score(Y_Test, clf.predict(X_Test)))
average_ROC(mean_fpr,tprs,aucs_NB,TP,TN,FP,FN)                               #plot average roc curve
average_performance(aucs_NB,Accuracy,TP,TN,FP,FN)                            #print the average performance of the model


## XGBoost

In [ ]:
Accuracy = []                                                                # for store the value of accuracy
FP = []                                                                      # for store False Positive
TN = []                                                                      # for True Negative
FN = []                                                                      # for False Negative
TP = []                                                                      # for True Positive
tprs = []                                                                    # for true positive rates
aucs_xboost = []                                                                # for store the values of auc
sn = []                                                                      # for sensitivity
sp = []                                                                      # for specificity
pr = []                                                                      # for precision
FOR = []                                                                     # for False omission rate
DOR = []                                                                     # for Diagnostic odds ratio (DOR)
iterator=0
mean_fpr = np.linspace(0, 1, 100)
fig = plt.figure(figsize=(8, 5))


for train_index, test_index in kf.split(X_Data,Y_Lavel):                     # split into train and test
#   print("TRAIN:", train_index, "TEST:", test_index)
    X_Train, X_Test = X_Data[train_index], X_Data[test_index]                #train data and label
    Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]              #test data and label

    #####################################
    ## define the parameters
    ######################################
    tuned_parameters = {
        'min_child_weight': [1, 5, 10],
        'gamma': [0.5, 1, 1.5, 2, 5],
        'subsample': [0.5, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'max_depth': [3, 4, 5]
        }

    clf = creat_Model (classifier = xgb.XGBClassifier(objective = "binary:logistic", eval_metric = 'error', random_state=random_initializer),
                      X_Train = X_Train,                                        # create model using XGB classifier
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters,
                      verbose=0)
    tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,               #evaluate the model parameters
                                                y_pred = clf.predict(X_Test),
                                                probas_ = clf.predict_proba(X_Test))
    tprs.append(interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    aucs_xboost.append(roc_auc)
    plot_Current_ROC(fpr,tpr,iterator,roc_auc)                                  #plot the roc of current fold
    iterator += 1
    TN.append(tn)
    FP.append(fp)
    FN.append(fn)
    TP.append(tp)
    Accuracy.append(accuracy_score(Y_Test, clf.predict(X_Test)))
    sn.append(tp/(tp+fn))
    sp.append(tn/(fp+tn))
    pr.append(tp/(tp+fp))
    FOR.append(fn/(tn+fn))
    DOR.append((tp*tn)/(fp*fn))

average_ROC(mean_fpr,tprs,aucs_xboost,TP,TN,FP,FN)                              #plot average roc curve
average_performance(aucs_xboost,Accuracy,TP,TN,FP,FN)                           #print the average performance of the model

#####################################################################
#    print the sensitivity,specificity,precision,for,dor of model
#####################################################################
print("Sensitivity (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(sn),np.std(sn)))
print("Specificity (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(sp),np.std(sp)))
print("Precision (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(pr),np.std(pr)))
print("FOR (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(FOR),np.std(FOR)))
print("DOR (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(DOR),np.std(DOR)))


## ensemble

### Helper function for Ensembling model

In [ ]:
def Ensembler(n_model, X_Train, Y_Train, X_Test, Y_Test):

    '''
    Parameters :
    Input -
    n_model =Number of models we need to ensemble
    X_Train =train data
    Y_Train =train data label
    X_Test =test data
    Y_Test =test data label
    weight =weight methods of models when ensembling


    Return -
    Returned a tuned ensembelled classifier using grid search

    '''

                                                                      ###### define the hyper parameters(number of neighbor,leaf_size,Distance) of KNN
    n_neighbors = [1,3,5,7,9,11,13,15,17,19,21,23,25,27,29,31,33,35,37,39,41,43,45,47,49]
    leaf_size = [5,10,15,20,25,30,35,40,45,50]
    Distance = [1,2]
                                                                      #define parameters with different algorithm
    tuned_parameters_knn = [ {'n_neighbors': n_neighbors,
                        'algorithm' : ['brute'],
                        'p':Distance},

                         {'n_neighbors': n_neighbors,
                        'algorithm' : ['ball_tree'],
                        'leaf_size' : leaf_size,
                        'p':Distance},

                        {'n_neighbors': n_neighbors,
                        'algorithm' : ['kd_tree'],
                        'leaf_size' : leaf_size,
                        'p':Distance}]

    clf_knn = creat_Model (classifier = KNeighborsClassifier(),       #create knn with utility function
                      X_Train = X_Train,
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters_knn,
                      verbose=0)

    tuned_parameters_ab = { 'algorithm': ['SAMME','SAMME.R'],         # define parameters of adaboost
                   'learning_rate':[0.1,0.5,1.0],
                   'n_estimators': [10,50,100,200]}

    clf_ab = creat_Model (classifier = AdaBoostClassifier(random_state=random_initializer),
                  X_Train = X_Train,                                 #create model with AdaBoost classifier with utility function
                  Y_Train = Y_Train,
                  tuned_parameters = tuned_parameters_ab,
                  verbose=0)

    tuned_parameters_rf = {'criterion': ['gini','entropy']}          # define parameters of RandomForestClassifier

    clf_rf = creat_Model (classifier = RandomForestClassifier(random_state=random_initializer),
                      X_Train = X_Train,                             #create model with RandomForest  classifier with utility function
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters_rf,
                      verbose=0)

    var_smoothing = [1e-01,
                    1e-02,
                    1e-03,
                    1e-04,
                    1e-05,
                    1e-06,
                    1e-07,
                    1e-08,
                    1e-09,
                    1e-10,
                    1e-11,
                    1e-12]

    tuned_parameters_nb = [{'var_smoothing': var_smoothing}]         # define parameters of Naive Bais

    clf_nb = creat_Model (classifier = GaussianNB(),                 #create model with Naive Bais  classifier with utility function
                      X_Train = X_Train,
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters_nb,
                      verbose=0)
    tuned_parameters_xb = {                                          # define parameters of XGBClassifier
        'min_child_weight': [1, 5, 10],
        'gamma': [0.5, 1, 1.5, 2, 5],
        'subsample': [0.5, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'max_depth': [3, 4, 5]
        }


    clf_xb = creat_Model (classifier = xgb.XGBClassifier(objective = "binary:logistic", eval_metric = 'error',random_state=random_initializer),
                      X_Train = X_Train,                              #create model with XGB Classifier classifier with utility function
                      Y_Train = Y_Train,
                      tuned_parameters = tuned_parameters_xb,
                      verbose=0)


    if n_model == 2:                                              #Using 2 best model we create ensembled model with soft voting
        model = VotingClassifier([('nb',clf_nb),
                                  ('knn', clf_knn)],
                                  voting='soft')
        model.fit(X_Train,Y_Train)
        return model
    if n_model == 3:                                               #Using 3 best model we create ensembled model with soft voting
        model = VotingClassifier([('nb',clf_nb),
                                  ('knn', clf_knn),
                                  ('ab', clf_ab)],
                                  voting='soft')
        model.fit(X_Train,Y_Train)
        return model
    if n_model == 4:                                               #Using 4 best model we create ensembled model with soft voting
        model = VotingClassifier([('nb',clf_nb),
                                  ('knn', clf_knn),
                                  ('ab', clf_ab),
                                  ('rf',clf_rf)],
                                  voting='soft')
        model.fit(X_Train,Y_Train)
        return model
    if n_model == 5:                                               #Using 5 best model we create ensembled model with soft voting
        model = VotingClassifier([('nb',clf_nb),
                                  ('knn', clf_knn),
                                  ('ab', clf_ab),
                                  ('rf',clf_rf),
                                  ('xb', clf_xb)],
                                  voting='soft')
        model.fit(X_Train,Y_Train)
        return model


### Ensembling model

In [ ]:
for i in range(2,6):
    Accuracy = []                                                                # for store the value of accuracy
    FP = []                                                                      # for store False Positive
    TN = []                                                                      # for True Negative
    FN = []                                                                      # for False Negative
    TP = []                                                                      # for True Positive
    tprs = []                                                                    # for true positive rates
    aucs_ens = []                                                                # for store the values of auc
    sn = []                                                                      # for sensitivity
    sp = []                                                                      # for specificity
    pr = []                                                                      # for precision
    FOR = []                                                                     # for False omission rate
    DOR = []                                                                     # for Diagnostic odds ratio (DOR)
    iterator=0
    mean_fpr = np.linspace(0, 1, 100)
    fig = plt.figure(figsize=(8, 5))

    ####################
    ##priniting solid line using +
    plus_print=n_dots*'+'
    print(plus_print)
    print('model running with ensembling  ---  '+str(i )+'  models')
    print(plus_print)
    ####################


    for train_index, test_index in kf.split(X_Data,Y_Lavel):                    # split data in train,test
        #   print("TRAIN:", train_index, "TEST:", test_index)
        X_Train, X_Test = X_Data[train_index], X_Data[test_index]               # the train data and label
        Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]             # the  test data and label

    ###################################################################
        clf = Ensembler( i, X_Train, Y_Train, X_Test, Y_Test)           #create ensemble model passing arguments

        tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,           #evaluation parameters of ensembelled model
                                                    y_pred = clf.predict(X_Test),
                                                    probas_ = clf.predict_proba(X_Test))
        tprs.append(interp(mean_fpr, fpr, tpr))
        tprs[-1][0] = 0.0
        aucs_ens.append(roc_auc)
        plot_Current_ROC (fpr,tpr,iterator,roc_auc)                             #plot the ROC curve of current fold
        iterator += 1
        TN.append(tn)
        FP.append(fp)
        FN.append(fn)
        TP.append(tp)
        Accuracy.append(accuracy_score(Y_Test, clf.predict(X_Test)))
        sn.append(tp/(tp+fn))
        sp.append(tn/(fp+tn))
        pr.append(tp/(tp+fp))
        FOR.append(fn/(tn+fn))
        DOR.append((tp*tn)/(fp*fn))

    average_ROC(mean_fpr,tprs,aucs_ens,TP,TN,FP,FN)                             #plot average ROC
    average_performance(aucs_ens,Accuracy,TP,TN,FP,FN)                          #print the average performance
    print("Sensitivity (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(sn),np.std(sn)))
    print("Specificity (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(sp),np.std(sp)))
    print("Precision (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(pr),np.std(pr)))
    print("FOR (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(FOR),np.std(FOR)))
    print("DOR (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(DOR),np.std(DOR)))


# 2-2

## data processing and plot

### Loading of different packagaes and APIs

In [ ]:
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Use CPU only


**python 3.11** <br>
**tensorflow 2.12** <br>
**keras 2.12** <br>


In [ ]:
import numpy as np
np.random.seed(6)
import random
random.seed(6)
import tensorflow as tf
import random
import numpy as np
# Set the seed for reproducibility
seed_value = 6
tf.random.set_seed(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
# sns.set(style="whitegrid")
import warnings
from sklearn.svm import SVC, NuSVC
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors  import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from scipy import stats
from scipy.stats import uniform, randint
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold, StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import roc_curve, auc, accuracy_score
# from tflearn.data_utils import to_categorical
from sklearn import preprocessing
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
from scipy import interp
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
from sklearn.decomposition import FastICA
from keras.utils import to_categorical
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV, KFold
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from keras.models import Sequential
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint
from keras.layers import Activation, Dense, Dropout, BatchNormalization, Input
from keras.models import Model
from keras.optimizers import Adam
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import VotingClassifier


In [ ]:
# import tensorflow as tf
# print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


### make the dataset file SAheart.csv

In [ ]:
import pandas as pd
import requests
from io import StringIO
import os

# Define the URL
url = "https://hastie.su.domains/ElemStatLearn/datasets/SAheart.data"

# Specify the path in your Google Drive
save_path = 'SAheart.csv'

# Check if the file already exists
if os.path.exists(save_path):
    # If it exists, read it as a DataFrame
    df = pd.read_csv(save_path)
    # Display the first five rows
    print("DataFrame from existing SAheart.csv:\n")
    df.head()
else:
    # If it doesn't exist, download the data, create DataFrame, and save it
    response = requests.get(url)
    data = response.text

    # Read the data into a DataFrame
    columns = [
        "row",
        "sbp",
        "tobacco",
        "ldl",
        "adiposity",
        "famhist",
        "typea",
        "obesity",
        "alcohol",
        "age",
        "chd",
    ]
    df = pd.read_csv(StringIO(data), sep=",", header=None, names=columns, na_values="NA")

    # Drop the first row
    df = df.iloc[1:]

    # Drop the first column
    df = df.iloc[:, 1:]

    encoder = OrdinalEncoder()

    # Reshape the 'famhist' column to a 2D array
    df['famhist'] = encoder.fit_transform(df['famhist'].values.reshape(-1, 1))

    # Save the DataFrame to a CSV file in Google Drive
    df.to_csv(save_path, index=False)

    # Display the DataFrame
    print("New DataFrame created and saved as SAheart.csv:\n")
    df.head()


### Utility Functions

In [ ]:
Renamed_feature= []               #list of names that will rename to feature column
all_clf_res=[]                    #every classifier auc values are stored in it
random_initializer=100            #random initializer
n_dots=50
##########################################################

for i in range(9):
  #for renaming dataset of columns features F1 -- F9
  Renamed_feature.append('F'+str(i+1))
############################################################

# Pairs plots are just showing all variables paired with all the other variables
def pair_plot(data):
  '''
  This function will create a grid of Axes such that each variable
  in data will by shared in the y-axis across a single row and in the x-axis
  across a single column.The diagonal Axes are treated differently, drawing
  a plot to show the univariate distribution of the data for the variable in
  that column.
  Parameters :
  Input - data is the pandas type variable for
  plotting pair plot of features in this
  dataframe

  Output :
  This function Plot pairwise relationships in a dataset.
  '''
  plt.figure()

  pair_plot =sns.pairplot(data=data,
                          height=3,
                          hue='chd',
                          diag_kind='kde')
  pair_plot.fig.suptitle("Pairplot of all features")
  plt.show()




###################################################################
# this function for Gaussian distribution plot
# and box plot simultaneously in a figure
def Box_Gaussian(data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output - The Gaussian distribution plot for
  eight feature of input data
  '''

  fig=plt.figure(figsize=(15,6))                                #define figure size
  fig.suptitle("Box Gaussian plot of all features")

  n_scaler = preprocessing.StandardScaler()                 #standardization function
  temp_Data = n_scaler.fit_transform(data)                  #pass into function for standrd.
  for i in range(9):                                        #loop for all 9 feature

    plt.subplot(3, 3, i+1)                                  #subplot for 3 rows in 3 columns
    Data = temp_Data[:,i]                                   #data for every feature
    sns.kdeplot(Data, shade=True,color='red', alpha=0.3)    #kernel density function under red shaded arae
    ax = sns.boxplot(Data, saturation=0.9, color="green")   #boxplot  with green shaded area
                                                            # https://seaborn.pydata.org/generated/seaborn.kdeplot.html
                                                            # https://seaborn.pydata.org/generated/seaborn.boxplot.html
    plt.gca().invert_yaxis()                                #Reverse Y-Axis in PyPlot
    # plt.title('F'+str(i+1))
    frame1 = plt.gca()
    frame1.axes.xaxis.set_ticklabels([])                    #removing xlabel data
    plt.ylim((-0.5,0.65))                                   #y axis  limit
    plt.tight_layout()                                      #This module provides routines to adjust subplot params so that subplots are nicely fit in the figure.
                                                            # https://matplotlib.org/api/tight_layout_api.html
    # plt.grid('on')

    for patch in ax.artists:
      r, g, b, a = patch.get_facecolor()                     #Get the facecolor of the Axes.
      patch.set_facecolor((r, g, b, 0.3))                    #set colour intensity
#############################################################


def plot_confusionMatrix(data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output -visualization of correalation matrix of
  input data
  '''
  sns.set(font_scale=1.15)                                    # Set aesthetic parameters in one step.
  ax = plt.figure(figsize=(10, 8))                            #set figure size   https://seaborn.pydata.org/generated/seaborn.set.html
  plt.title("Confusion Matrix of all features")
  sns.heatmap(data.corr(),                                    # input correlation matrix  of dataset
              vmax=1.0,                                       #Values to anchor the colormap, otherwise they are inferred from
                                                              #the data and other keyword arguments.
              vmin=0.0,
              linewidths=0.01,
              square=False,                                   #If True, set the Axes aspect to “equal” so each cell will be square-shaped.
              annot=True,                                     #If True, write the data value in each cell.
              linecolor="black")                              #Color of the lines that will divide each cell.
                                                              #https://seaborn.pydata.org/generated/seaborn.heatmap.html
  b, t = plt.ylim()                                           # discover the values for bottom and top
  b += 0.5                                                    # Add 0.5 to the bottom
  t -= 0.5                                                    # Subtract 0.5 from the top
  plt.ylim(b, t)                                              # update the ylim(bottom, top) values
  plt.show()



############################################################
# this function plot univariate distribution of
# every feature

def dist_Plot(data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output - The distribution plot for
  eight feature of input data
  '''
  fig, ax = plt.subplots(3,3, figsize=(12,5))                 #set numbers of rows and columns of subplot
  sns.set()
  sns.distplot(data.F1, bins = 10, ax=ax[0,0])                #Flexibly plot a univariate distribution of observations.
  sns.distplot(data.F2, bins = 10, ax=ax[0,1])
  sns.distplot(data.F3, bins = 10, ax=ax[0,2])
  sns.distplot(data.F4, bins = 10, ax=ax[1,0])
  sns.distplot(data.F5, bins = 10, ax=ax[1,1])
  sns.distplot(data.F6, bins = 10, ax=ax[1,2])
  sns.distplot(data.F7, bins = 10, ax=ax[2,0])
  sns.distplot(data.F8, bins = 10, ax=ax[2,1])
  sns.distplot(data.F9, bins = 10, ax=ax[2,2])
  fig.suptitle("Gaussian Distribution of all features")
  fig.tight_layout()                                          #This module provides routines to adjust subplot params
                                                              #  so that subplots are nicely fit in the figure.


############################################################
# this function plot violin plot  of
# every feature


def plot_violinplot (data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Output - The violinplot plot for
  eight feature of input data
  '''

  # A violin plot is a method of plotting numeric data.
  # It is similar to box plot with a rotated kernel
  # density plot on each side. Violin plots are similar
  # to box plots, except that they also show the probability
  # density of the data at different values (in the simplest
  # case this could be a histogram).
  fig, ax = plt.subplots(3,3, figsize=(12,6))
  # #set numbers of rows and columns of subplot and figure size
  sns.set()
  sns.violinplot(x = data.chd, y=data.F1,  ax=ax[0,0])    #violine plot for F1 feature
  sns.violinplot(x = data.chd, y=data.F2,  ax=ax[0,1])    #violine plot for F2 feature
  sns.violinplot(x = data.chd, y=data.F3,  ax=ax[0,2])    #violine plot for F3 feature
  sns.violinplot(x = data.chd, y=data.F4,  ax=ax[1,0])    #violine plot for F4 feature
  sns.violinplot(x = data.chd, y=data.F5,  ax=ax[1,1])    #violine plot for F5 feature
  sns.violinplot(x = data.chd, y=data.F6,  ax=ax[1,2])    #violine plot for F6 feature
  sns.violinplot(x = data.chd, y=data.F7,  ax=ax[2,0])    #violine plot for F7 feature
  sns.violinplot(x = data.chd, y=data.F8,  ax=ax[2,1])    #violine plot for F8 feature
  sns.violinplot(x = data.chd, y=data.F9,  ax=ax[2,2])    #violine plot for F9 feature
  fig.suptitle("Violin plot of all features")
  fig.tight_layout()

                                                              # https://seaborn.pydata.org/generated/seaborn.violinplot.html

############################################################

# this function if for outlair rejection with
# respect to mean value
def IQR_Mean (data):

  '''
  Parameters :
  Input - data is the pandas type variable

  Return - dataframe with outleir rejection
  filled with mean
  of input data
  '''
  for i in range(9):
    x = data[Renamed_feature[i]]
    Q1 = x.quantile(0.25)                                   # Q1 is the "middle" value in the first half of the rank-ordered data set.
    Q3 = x.quantile(0.75)                                   # Q3 is the "middle" value in the second half of the rank-ordered data set.
    IQR = Q3-Q1                                             # The interquartile range is equal to Q3 minus Q1.
    mean = x.mean()                                         #mean of feature
    for j in range(462):                                    # loop for first 462 elements of feature
      temp = x[j]                                           # every feature value
      LW = (Q1 - 1.5 * IQR)                                 #lower considerable range of gaussian distribution
      UW = (Q3 + 1.5 * IQR)                                 #upper considerable range of gaussian distribution
      if temp < LW:                                         #replace upper value with mean
        x[j] = mean
      if temp > UW:                                         #replace lower value with mean
        x[j] = mean
    data[Renamed_feature[i]] = x
  return data

############################################################
# this function if for outlair rejection with
# respect to median value same as previous function
def IQR_Medain (data):
  '''
  Parameters :
  Input - data is the pandas type variable

  Return - dataframe with outleir rejection
  filled with median of input data
  '''
  for i in range(9):
    x = data[Renamed_feature[i]]
    Q1 = x.quantile(0.25)
    Q3 = x.quantile(0.75)
    IQR = Q3-Q1
    median = x.quantile(0.5)                                # find the median
    for j in range(462):                                    #replace the first 462 values with respect to median
      temp = x[j]
      LW = (Q1 - 1.5 * IQR)
      UW = (Q3 + 1.5 * IQR)
      if temp < LW:                                         #replace upper value with median
        x[j] = median
      if temp > UW:
        x[j] = median                                       #replace upper value with median
    data[Renamed_feature[i]] = x
  return data

############################################################



# this function if for outlair rejection with
# 1.5 times of IQR that means that are
# significant in gaussian distribution
def IQR (data):

  '''
  Parameters :
  Input - data is the pandas type variable

  Return - dataframe with outleir rejection
  of input data

  '''
  #input dataset as data
  for i in range(9):                                        # for every feature
    Q1 = data[Renamed_feature[i]].quantile(0.25)
    Q3 = data[Renamed_feature[i]].quantile(0.75)
    IQR = Q3-Q1                                             #find IQR
    LW = (Q1 - 1.5 * IQR)                                   #find lower boundary
          # print(LW)
    UW = (Q3 + 1.5 * IQR)                                   #find upper boundary
          # print(UW)
    data = data[data[Renamed_feature[i]]<UW]                #drop greater than upper limit
    data = data[data[Renamed_feature[i]]>LW]                #drop smaller than lower limit

  return data


############################################################
#outlier rejection with different condition

def outlier_Rejection (data, iqr_Mean, iqr_Medain, iqr):
  '''
  Parameters :
  Input -
  data is the pandas type variable
  iqr_Mean - for outleir rejection with Mean
  iqr_Medain- for outleir rejection with Medain
  iqr- for drop the outleir
  manual -for manual rejection
  Return - dataframe with outleir rejection
  filled with Input parameter

  '''

  # outlier_Rejection with conditional input
  if iqr_Mean == True:                                     #reject outleir with Mean
    data = IQR_Mean (data)
  if iqr_Medain == True:                                   #reject outleir with Median
    data = IQR_Medain (data)
  if iqr == True:                                          #reject outleir in IQR range
    data = IQR (data)

  return data

############################################################

#data plot on different input condition
def data_plot (data,
               Pair_plot,
               Dist_Plot,
               Plot_violinplot,
               Plot_confusionMatrix,
               box_Gaussian):

  '''
  Parameters :
  Input -
  data - It is the pandas type variable
  Pair_plot - for pair plot visualization of input  data
  Dist_Plot- for gaussian distribution plot visualization of input  data
  Plot_violinplot- for violin plot visualization of input  data
  Plot_confusionMatrix -for confusion matrix visualization of input  data

  Output - dataframe with outleir rejection
  filled with Input parameter

  '''
  if Pair_plot ==True:
    pair_plot(data)

  if Dist_Plot ==True:
    dist_Plot(data)

  if Plot_violinplot ==True:
    plot_violinplot (data)

  if Plot_confusionMatrix ==True:
    plot_confusionMatrix(data)

  if box_Gaussian ==True:
    Box_Gaussian(data)


############################################################


def metrics (y_true, y_pred, probas_):


  '''
  Parameters :
  Input -
  y_true - true  value of input data
  y_pred- predicted  value of input data
  probas_- probability/confidence of predicted output

  return -True Negative(tn),False Positive(fp),False Negative(fn)
  True positive(tp),AUC(roc_auc),False Positive Rate(fpr),
  True positive rate(tpr)

  '''


  points=n_dots*'-'
  print(points)
#    print("Best parameters set found on development set:")
#    print(clf.best_params_)
  fpr, tpr, thresholds = roc_curve(y_true, probas_[:, 1])
  tprs.append(interp(mean_fpr, fpr, tpr))
  tprs[-1][0] = 0.0
  roc_auc = auc(fpr, tpr)
  #  aucs.append(roc_auc)
  print("Detailed classification report for current fold:")
  print()
  print(classification_report(y_true, y_pred))
  print()
  print("Area Under ROC (AUC): {}".format(roc_auc))
  print()
  print('Confusion Matrix for current fold: ')
  print(confusion_matrix(y_true, y_pred))
  print()
  print("Accuracy for Current Fold: {}".format(accuracy_score(y_true, y_pred)))
  print()
  tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

  return  tn, fp, fn, tp, roc_auc, fpr, tpr

############################################################


def average_ROC(mean_fpr,tprs,aucs,TP,TN,FP,FN):

  '''
  Parameters :
  mean_fpr - Mean False positive rate
  tprs -values of true positive rate
  aucs  - values of auc
  TP    - True positive
  TN    - True Negative
  FP    - False Positiv
  FN    - False Negative

  Output -
  Visalization of TPR vs FPR plot
  '''
  sen = (np.sum(TP))/(np.sum(TP)+np.sum(FN))
  spe = (np.sum(TN))/(np.sum(TN)+np.sum(FP))

  mean_tpr = np.mean(tprs, axis=0)
  mean_tpr[-1] = 1.0
  # mean_auc = auc(mean_fpr, mean_tpr)
  mean_auc = np.mean(aucs)
  std_auc = np.std(aucs)
  # plt.figure(figsize=(8, 5))
  # plt.grid(True)
  ax = plt.axes()
  ax.grid(color='lightgray', linestyle='-', linewidth=.5)
  # Setting the background color
  ax.set_facecolor("white")

  ax.spines['bottom'].set_color('#000000')
  ax.spines['top'].set_color('#000000')
  ax.spines['right'].set_color('#000000')
  ax.spines['left'].set_color('#000000')

  plt.plot(mean_fpr, mean_tpr, color='blue',
          label=r'Avg. ROC (AUC (avg $\pm$ std) = %0.3f $\pm$ %0.3f)' % (mean_auc, std_auc),
          lw=2, alpha=.8)

  plt.scatter((1-spe), sen, s=80, c='r', marker='x',)
  plt.scatter(0, sen, s=80, c='r', marker='x',)
  plt.scatter((1-spe),0, s=80, c='r', marker='x',)
  plt.axhline(y=sen, color='r', linestyle='--')
  plt.axvline(x=(1-spe), color='r', linestyle='--')
  plt.text((1-spe), 0.02, "FPR={:0.3f}".format((1-spe)))
  plt.text(0.009, sen+0.05, "TPR={:0.3f}".format(sen))

  std_tpr = np.std(tprs, axis=0)
  tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
  tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
  plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='darkgray', alpha=0.5,
                  label=r'$\pm$ 1 Standard deviation')

  plt.xticks(np.arange(0.0, 1.01, step=0.1))
  plt.yticks(np.arange(0.0, 1.01, step=0.1))
  left=0.0
  right=1.0
  plt.xlim(left, right)
  plt.ylim(left, right)
  plt.xlabel('False Positive Rate (FPR)')
  plt.ylabel('True Positive Rate (TPR)')
  plt.legend(loc="lower right")
  # plt.grid(True)
  plt.show()

############################################################

def plot_Current_ROC(fpr,tpr,iterator,roc_auc):

  '''
  Parameters :
  Input -
  fpr - False positive rate
  tpr - True positive rate
  roc_auc -auc values of roc curve

  Output -
  Visalization of current roc curve

  '''
  plt.plot(fpr,

          tpr,
          # Color[iterator],
          alpha=0.35,
          # label='macro-average ROC (AUC = {0:0.3f})'.format(roc_auc)
          # +FOLD[iterator],
          linewidth=1)

############################################################


def creat_Model (classifier, X_Train, Y_Train, tuned_parameters, verbose):

  '''
  Parameters :
  Input -
  X_Train -train data
  Y_Train - label/output of train data
  tuned_parameters =parameters of models
  verbose = condition about model

  Output -
  Returned a tuned classifier using grid search
  '''
  clf = GridSearchCV(classifier,
                    tuned_parameters,
                    verbose=verbose,
                    cv=5,
                    scoring='roc_auc',
                    n_jobs=None)
  clf.fit(X_Train, Y_Train)
  return clf
############################################################

def average_performance(aucs,Accuracy,TP,TN,FP,FN):

  '''
  Parameters :
  Input -
  aucs= values of aucs
  Accuracy - value of accuracy
  TP  - True Positive
  TN  - True Negative
  FP  - False Positive
  FN  - False Negative


  Output -
  It prints the average aucs,accuarcy,confusion matrix
  '''

  print()
  n_dotsav=(n_dots-len('Average'))//2

  print('-'*n_dotsav+'Average'+'-'*n_dotsav)
  print("AUC (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(aucs),np.std(aucs)))
  print("Accuracy (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(Accuracy),np.std(Accuracy)))
  cm = [[int(np.mean(TP)), int(np.mean(FP))],[int(np.mean(FN)), int(np.mean(TN))]]
  print ('Avg. CM is '+str(cm))
  cm = [[int(np.sum(TP)), int(np.sum(FP))],[int(np.sum(FN)), int(np.sum(TN))]]
  print ('Total for all folds CM is '+str(cm))
  re_auc=str(round(np.mean(aucs), 3))+'+/-'+str(round(np.std(aucs),3))
  all_clf_res.append(re_auc)

############################################################

#this  function is for algorithm based feature selection
def feature_Selector(data, algo, n_feature):
    '''
    Parameters :
    Input -
    data - It is the pandas type variable
    algo - type of algorith PCA,ICA,Correlation


    Output -
    It prints the average aucs,accuarcy,confusion matrix
    '''
    if algo=='PCA':                                                   #for pca algorithm
        X_Data= data.iloc[:,:9].values
        pca = PCA(n_components=n_feature)                             #number of feature
        X_Data = pca.fit_transform(X_Data)
        return X_Data , data.iloc[:,9:].values

    if algo == 'ICA':
        X_Data= data.iloc[:,:9].values
        ICA = FastICA(n_components=n_feature, random_state=12)
        X_Data = ICA.fit_transform(X_Data)
        return X_Data , data.iloc[:,9:].values

    if algo =='corr':                                                   #for ica algorithm
        if n_feature ==4:
            data = data[['F2','F5','F4','F6','Outcome']]                #for 4 feature
            return data.iloc[:,:4].values, data.iloc[:,4:].values
        if n_feature ==6:
            data = data[['F1','F2','F4','F5','F6','F8','Outcome']]       #for 6 feature
            return data.iloc[:,:6].values, data.iloc[:,6:].values

    if algo == 'None':
        return data.iloc[:,:9].values, data.iloc[:,9:].values            #if feature selection is off all features are counted



def run (hLayer,
         batch_size = [32],
         epochs = [100],
         learn_rate = [0.001],
         dropout_rate = [0.3],
         activation = ['relu'],
         init =['normal']):
    """
    Input : hidden layer number
    and others are alwalys constant

    Output : optimized model with best number of neuron
    in every layer
    """


    if hLayer==1:

        ###########################################
        #  number of neurons  for every layer
        #  optimization
        neuron1 = [16, 32, 64]
        neuron2 = [16, 32, 64]
        ############################################


        #############################################
        # parameter dictionary for grid search#
        param_grid = dict(batch_size=batch_size,
                          epochs=epochs,
                          learn_rate=learn_rate,
                          dropout_rate=dropout_rate,
                          activation=activation,
                          init=init,
                          neuron1=neuron1,
                          neuron2=neuron2)
        ###############################################

        ###############################################
        # model building function for given parameters#
        def nn_opt_1(activation,
                   dropout_rate,
                   init,
                   learn_rate,
                   neuron1,
                   neuron2):
            """
            Input : activation ,dropout_rate,init,learn_rate,neuron1,neuron2

            Output : Using input hyper_parameters build model and return it
            """

            #####################################################################
            ##               model initialization and building block           ##
            model = Sequential()
            model.add(Dense(neuron1, input_dim = 6, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron2, input_dim = neuron1, kernel_initializer= init, activation= activation))
            model.add(Dropout(dropout_rate))
            model.add(Dense(2, activation='softmax'))
            optimizer = Adam(learning_rate = learn_rate)
            model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

            #####################################################################

            return model

        ########################### grid search for optimisaion #############################
        grid = GridSearchCV(estimator = KerasClassifier(build_fn = nn_opt_1, verbose = 0),
                            param_grid = param_grid,
                            cv = 5,
                            n_jobs = None,
                            verbose = 1)
        grid_results = grid.fit(X_Data, to_categorical(Y_Lavel,2))

        ########################################################################################

        ########################################################################################
        ##                  build model with optimized parameters                             ##
        model =nn_opt_1(grid_results.best_params_['activation'],
                grid_results.best_params_['dropout_rate'],
                grid_results.best_params_['init'],
                grid_results.best_params_['learn_rate'],
                grid_results.best_params_['neuron1'],
                grid_results.best_params_['neuron2'])
        ########################################################################################

        #  return best parameters and model
        return grid_results.best_params_['batch_size'], grid_results.best_params_['epochs'], model, grid_results


    if hLayer==2:
        neuron1 = [16, 32, 64]
        neuron2 = [16, 32, 64]
        neuron3 = [16, 32, 64]
        param_grid = dict(batch_size=batch_size,
                          epochs=epochs,
                          learn_rate=learn_rate,
                          dropout_rate=dropout_rate,
                          activation=activation,
                          init=init,
                          neuron1=neuron1,
                          neuron2=neuron2,
                          neuron3=neuron3)

        def nn_opt_2(activation,
                   dropout_rate,
                   init,
                   learn_rate,
                   neuron1,
                   neuron2,
                   neuron3):
            model = Sequential()
            model.add(Dense(neuron1, input_dim = 6, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron2, input_dim = neuron1, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron3, input_dim = neuron2, kernel_initializer= init, activation= activation))
            model.add(Dropout(dropout_rate))
            model.add(Dense(2, activation='softmax'))
            optimizer = Adam(learning_rate = learn_rate)
            model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
            return model

        grid = GridSearchCV(estimator = KerasClassifier(build_fn = nn_opt_2, verbose = 0),
                            param_grid = param_grid,
                            cv = 5,
                            n_jobs = None,
                            verbose = 1)
        grid_results = grid.fit(X_Data, to_categorical(Y_Lavel,2))

        model =nn_opt_2(grid_results.best_params_['activation'],
                grid_results.best_params_['dropout_rate'],
                grid_results.best_params_['init'],
                grid_results.best_params_['learn_rate'],
                grid_results.best_params_['neuron1'],
                grid_results.best_params_['neuron2'],
                grid_results.best_params_['neuron3'])
        return grid_results.best_params_['batch_size'], grid_results.best_params_['epochs'], model, grid_results

    if hLayer==3:
        neuron1 = [16, 32, 64]
        neuron2 = [16, 32, 64]
        neuron3 = [16, 32, 64]
        neuron4 = [16, 32, 64]
        param_grid = dict(batch_size=batch_size,
                          epochs=epochs,
                          learn_rate=learn_rate,
                          dropout_rate=dropout_rate,
                          activation=activation,
                          init=init,
                          neuron1=neuron1,
                          neuron2=neuron2,
                          neuron3=neuron3,
                          neuron4=neuron4)

        def nn_opt_3(activation,
                   dropout_rate,
                   init,
                   learn_rate,
                   neuron1,
                   neuron2,
                   neuron3,
                   neuron4):
            model = Sequential()
            model.add(Dense(neuron1, input_dim = 6, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron2, input_dim = neuron1, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron3, input_dim = neuron2, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron4, input_dim = neuron3, kernel_initializer= init, activation= activation))
            model.add(Dropout(dropout_rate))
            model.add(Dense(2, activation='softmax'))
            optimizer = Adam(learning_rate = learn_rate)
            model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
            return model

        grid = GridSearchCV(estimator = KerasClassifier(build_fn = nn_opt_3, verbose = 0),
                            param_grid = param_grid,
                            cv = 5,
                            n_jobs = None,
                            verbose = 1)
        grid_results = grid.fit(X_Data, to_categorical(Y_Lavel,2))

        model =nn_opt_3(grid_results.best_params_['activation'],
                grid_results.best_params_['dropout_rate'],
                grid_results.best_params_['init'],
                grid_results.best_params_['learn_rate'],
                grid_results.best_params_['neuron1'],
                grid_results.best_params_['neuron2'],
                grid_results.best_params_['neuron3'],
                grid_results.best_params_['neuron4'])
        return grid_results.best_params_['batch_size'], grid_results.best_params_['epochs'], model, grid_results

    if hLayer==4:
        neuron1 = [16, 32, 64]
        neuron2 = [16, 32, 64]
        neuron3 = [16, 32, 64]
        neuron4 = [16, 32, 64]
        neuron5 = [16, 32, 64]
        param_grid = dict(batch_size=batch_size,
                          epochs=epochs,
                          learn_rate=learn_rate,
                          dropout_rate=dropout_rate,
                          activation=activation,
                          init=init,
                          neuron1=neuron1,
                          neuron2=neuron2,
                          neuron3=neuron3,
                          neuron4=neuron4,
                          neuron5=neuron5)

        def nn_opt_4(activation,
                   dropout_rate,
                   init,
                   learn_rate,
                   neuron1,
                   neuron2,
                   neuron3,
                   neuron4,
                   neuron5):
            model = Sequential()
            model.add(Dense(neuron1, input_dim = 6, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron2, input_dim = neuron1, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron3, input_dim = neuron2, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron4, input_dim = neuron3, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron5, input_dim = neuron4, kernel_initializer= init, activation= activation))
            model.add(Dropout(dropout_rate))
            model.add(Dense(2, activation='softmax'))
            optimizer = Adam(learning_rate = learn_rate)
            model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
            return model

        grid = GridSearchCV(estimator = KerasClassifier(build_fn = nn_opt_4, verbose = 0),
                            param_grid = param_grid,
                            cv = 5,
                            n_jobs = None,
                            verbose = 1)
        grid_results = grid.fit(X_Data, to_categorical(Y_Lavel,2))

        model =nn_opt_4(grid_results.best_params_['activation'],
                grid_results.best_params_['dropout_rate'],
                grid_results.best_params_['init'],
                grid_results.best_params_['learn_rate'],
                grid_results.best_params_['neuron1'],
                grid_results.best_params_['neuron2'],
                grid_results.best_params_['neuron3'],
                grid_results.best_params_['neuron4'],
                grid_results.best_params_['neuron5'])
        return grid_results.best_params_['batch_size'], grid_results.best_params_['epochs'], model, grid_results

    if hLayer==5:
        neuron1 = [16, 32, 64]
        neuron2 = [16, 32, 64]
        neuron3 = [16, 32, 64]
        neuron4 = [16, 32, 64]
        neuron5 = [16, 32, 64]
        neuron6 = [16, 32, 64]
        param_grid = dict(batch_size=batch_size,
                          epochs=epochs,
                          learn_rate=learn_rate,
                          dropout_rate=dropout_rate,
                          activation=activation,
                          init=init,
                          neuron1=neuron1,
                          neuron2=neuron2,
                          neuron3=neuron3,
                          neuron4=neuron4,
                          neuron5=neuron5,
                          neuron6=neuron6)

        def nn_opt_5(activation,
                   dropout_rate,
                   init,
                   learn_rate,
                   neuron1,
                   neuron2,
                   neuron3,
                   neuron4,
                   neuron5,
                   neuron6):
            model = Sequential()
            model.add(Dense(neuron1, input_dim = 6, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron2, input_dim = neuron1, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron3, input_dim = neuron2, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron4, input_dim = neuron3, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron5, input_dim = neuron4, kernel_initializer= init, activation= activation))
            model.add(Dense(neuron6, input_dim = neuron5, kernel_initializer= init, activation= activation))
            model.add(Dropout(dropout_rate))
            model.add(Dense(2, activation='softmax'))
            optimizer = Adam(learning_rate = learn_rate)
            model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
            return model

        grid = GridSearchCV(estimator = KerasClassifier(build_fn = nn_opt_5, verbose = 0),
                            param_grid = param_grid,
                            cv = 5,
                            n_jobs = None,
                            verbose = 1)
        grid_results = grid.fit(X_Data, to_categorical(Y_Lavel,2))

        model =nn_opt_5(grid_results.best_params_['activation'],
                grid_results.best_params_['dropout_rate'],
                grid_results.best_params_['init'],
                grid_results.best_params_['learn_rate'],
                grid_results.best_params_['neuron1'],
                grid_results.best_params_['neuron2'],
                grid_results.best_params_['neuron3'],
                grid_results.best_params_['neuron4'],
                grid_results.best_params_['neuron5'],
                grid_results.best_params_['neuron6'])
        return grid_results.best_params_['batch_size'], grid_results.best_params_['epochs'], model, grid_results







################################################################
#           this function optimize four hyper-parameters are
#           activation,dropout_rate,init,learn_rate            #
def nn_opt(activation,dropout_rate,init,learn_rate):
    """
  Parameters :
  Input - list of 4 hyper-parameter that are need to
  be optimized

  Output - Best optimized model
    """

    #define the optimmized neuron  number from experiment
    neuron1,neuron2,neuron3,neuron4,neuron5=32,64,64,64,64
    ###############################################################################################
    # the model building block

    model = Sequential()
    np.random.seed(6)
    model.add(Dense(neuron1, input_dim =6 , kernel_initializer= init, activation= activation))
    model.add(Dense(neuron2, input_dim = neuron1, kernel_initializer= init, activation= activation))
    model.add(Dense(neuron3, input_dim = neuron2, kernel_initializer= init, activation= activation))
    model.add(Dense(neuron4, input_dim = neuron3, kernel_initializer= init, activation= activation))
    model.add(Dense(neuron5, input_dim = neuron4, kernel_initializer= init, activation= activation))
    model.add(Dropout(dropout_rate))
    model.add(Dense(2, activation='softmax'))

    optimizer = Adam(learning_rate = learn_rate)               #optimizer of Neural network
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) #compile model
    #################################################################################################
    return model




### Read the data from the drive using pandas (Python Data Analysis Library)


In [ ]:
if colab ==True:
  data = pd.read_csv(data_dir)
else:
  data = pd.read_csv(data_dir)
data.shape


In [ ]:
data.head(n=6)


### Renaming the Features by F1, F2, and so on ......
---
New Name | Original Name | Comments|
---|---|---|
F1|sbp|systolic blood pressure|
F2|tobacco|cumulative tobacco (kg)|
F3|ldl|low densiity lipoprotein cholesterol|
F4|adiposity| |
F5|famhist|family history of heart disease (Present = 1, Absent = 0)|
F6|typea|type-A behavior|
F7|obesity| |
F8|alcohol|current alcohol consumption|
F9|age|age at onset|


In [ ]:
data = pd.DataFrame({'F1':data.iloc[:,:9].values[:,0],
                     'F2':data.iloc[:,:9].values[:,1],
                     'F3':data.iloc[:,:9].values[:,2],
                     'F4':data.iloc[:,:9].values[:,3],
                     'F5':data.iloc[:,:9].values[:,4],
                     'F6':data.iloc[:,:9].values[:,5],
                     'F7':data.iloc[:,:9].values[:,6],
                     'F8':data.iloc[:,:9].values[:,7],
                     'F9':data.iloc[:,:9].values[:,8],
                     'chd':data.iloc[:,9:].values[:,0]})


In [ ]:
data.describe()


### Raw Data Plot and Presenation

In [ ]:
data_plot (data,
          Pair_plot=True,
          Dist_Plot=True,
          Plot_violinplot=True,
          Plot_confusionMatrix=True,
          box_Gaussian=False)


### Data Preprocessing  

In [ ]:
print('Shape Before Process: ' + str(data.shape))
##########################################################################

## The process for the outlier rejection (P)

data = outlier_Rejection (data,
                  iqr_Mean=False,
                  iqr_Medain=False,
                  iqr=True)
print('Shape After outlier Removed: ' + str(data.shape))

##########################################################################
#  algo parameters are
# 'PCA','ICA','corr','None'

X_Data,Y_Lavel = feature_Selector(data, algo='PCA', n_feature=6)
print('Shape After Feature Selection: ' + str(X_Data.shape))


##########################################################################
# The process of Standardization  (S)
scaler =  preprocessing.StandardScaler()
X_Data,Y_Lavel= scaler.fit_transform(X_Data), Y_Lavel
print('Shape After Standardization: ' + str(X_Data.shape))


##########################################################################
# Stratified K-Folds cross-validator
# Provides train/test indices to split
# data in train/test sets.This cross-validation
#  object is a variation of KFold that returns
#  stratified folds. The folds are made by preserving
#  the percentage of samples for each class.

kf = StratifiedKFold(n_splits=5,
                     shuffle=False)


In [ ]:
print(X_Data.shape)
print(Y_Lavel.shape)


### Processed Data Plot and Presenation

In [ ]:
data_plot (data,
          Pair_plot=True,
          Dist_Plot=True,
          Plot_violinplot=True,
          Plot_confusionMatrix=True,
          box_Gaussian=False)


### MLP Experiment


### finding the optimum number of hiden layers and number of neurons

In [ ]:
################################
# list for store accuracy and auc value and plot
accuracy = []
AUC = []
##############################


################################################################################
# in this block we  optimize number of hiiden layer and neuron number
for hLayer in range(1,6):
    print ('------------------------------Model Running with # of Hidden = '+str(hLayer)+'-----------------------------')
    batch_size, epochs, model, grid_results = run (hLayer,
                                     batch_size = [32],
                                     epochs = [100],
                                     learn_rate = [0.001],
                                     dropout_rate = [0.3],
                                     activation = ['relu'],
                                     init =['normal'])

    print(grid_results.best_params_)

    Y_Train_1Hot = to_categorical(Y_Lavel,2)
    model.fit(x=X_Data,
            y=Y_Train_1Hot,
            batch_size=batch_size,
            epochs=epochs,
            verbose=0)
    probas_ = model.predict(X_Data)
    y_pred = np.argmax(model.predict(X_Data), axis=1)

    #############################################################################
    #   print and stroe the optimized model accuracy and  auc
    print(accuracy_score(Y_Lavel, y_pred))
    print(roc_auc_score(Y_Train_1Hot, probas_))
    accuracy.append(accuracy_score(Y_Lavel, y_pred))
    AUC.append(roc_auc_score(Y_Train_1Hot, probas_))
    #############################################################################


######   plot the value of auc and accuracy ####
plt.plot(AUC)
plt.plot(accuracy)
################################################


### finding the optimum hyper-parameters


In [ ]:
##########################
# Define a random seed
seed = 6
np.random.seed(seed)
###########################



# create the model for optimization
model = KerasClassifier(build_fn = nn_opt, verbose = 0)

##########################################################
##List of hyper-parameters of MLP model for optimization##

batch_size = [8, 16, 32]
epochs = [50, 100, 120, 200]
learn_rate =[0.001, 0.01, 0.05, 0.1, 0.5]
dropout_rate = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
activation = ['relu', 'LeakyReLU', 'tanh', 'ELU']
init =['uniform', 'normal']

###########################################################

###########################################################
## parameter dictionary for grid ssearch   ##
param_grid = dict(batch_size=batch_size,
                  epochs=epochs,
                  learn_rate=learn_rate,
                  dropout_rate=dropout_rate,
                  activation=activation,
                  init=init)

#build and fit the GridSearchCV
grid = GridSearchCV(estimator = model,
                    param_grid = param_grid,
                    cv = 5,
                    n_jobs = None,
                    verbose = 1)
###########################################################


In [ ]:
i=0                                                                          # for verification of fold number
Accuracy = []                                                                # for store the value of accuracy
FP = []                                                                      # for store False Positive
TN = []                                                                      # for True Negative
FN = []                                                                      # for False Negative
TP = []                                                                      # for True Positive
tprs = []                                                                    # for true positive rates
aucs_ens = []                                                                # for store the values of auc
sn = []                                                                      # for sensitivity
sp = []                                                                      # for specificity
pr = []                                                                      # for precision
FOR = []                                                                     # for False omission rate
DOR = []
iterator=0
mean_fpr = np.linspace(0, 1, 100)


########################################################

#      call grid  search for optimization       ##

grid_results = grid.fit(X_Data, to_categorical(Y_Lavel,2))

#            tuned/optimized parameters        ##

activation=grid_results.best_params_['activation']
batch_size=grid_results.best_params_['batch_size']
epochs=grid_results.best_params_['epochs']
learn_rate=grid_results.best_params_['learn_rate']
dropout_rate=grid_results.best_params_['dropout_rate']
init=grid_results.best_params_['init']

#########################################################


# ##########################################################
# ##     best optimizers found  are defined here by experiment ##
# activation='ELU'
# batch_size=8
# epochs=200
# learn_rate=.001
# dropout_rate=0.6
# init="normal"
# ############################################################
neuron1,neuron2,neuron3,neuron4,neuron5=32,64,64,64,64

print(activation,batch_size,epochs,learn_rate,dropout_rate,init,
    neuron1,neuron2,neuron3,neuron4,neuron5)



for train_index, test_index in kf.split(X_Data,Y_Lavel):                  # for k fold experiment
  print('------------------->>>>>>>>>>Fold no = ',i+1)
  X_Train, X_Test = X_Data[train_index], X_Data[test_index]               # the train data and label
  Y_Train, Y_Test = Y_Lavel[train_index], Y_Lavel[test_index]             # the  test data and label


  Y_Train_1Hot = to_categorical(Y_Train,2)                                #convert train output to catagorical
  Y_Test_1Hot = to_categorical(Y_Test,2)                                  #convert test output to catagorical

  model =nn_opt(activation,                                               #build model using tuned parameters
            dropout_rate,
            init,
            learn_rate)
  np.random.seed(6)
  model.fit(x=X_Train,                                                     # fit our model
            y=Y_Train_1Hot,
            batch_size=batch_size,
            epochs=epochs,
            shuffle=False,
            verbose=1)

  probas_ = model.predict(X_Test)                                           #predict the class probability

  y_pred = np.argmax(model.predict(X_Test), axis=1)                         #find the max. probability of output class


  tn, fp, fn, tp, roc_auc, fpr, tpr = metrics (y_true = Y_Test,           #evaluation parameters of ensembelled model
                                              y_pred = y_pred,
                                              probas_ = probas_)
  tprs.append(interp(mean_fpr, fpr, tpr))
  tprs[-1][0] = 0.0
  aucs_ens.append(roc_auc)
  plot_Current_ROC (fpr,tpr,iterator,roc_auc)                             #plot the ROC curve of current fold
  iterator += 1
  TN.append(tn)
  FP.append(fp)
  FN.append(fn)
  TP.append(tp)
  Accuracy.append(accuracy_score(Y_Test, y_pred))
  sn.append(tp/(tp+fn))
  sp.append(tn/(fp+tn))
  pr.append(tp/(tp+fp))
  FOR.append(fn/(tn+fn))
  DOR.append((tp*tn)/(fp*fn))
  print((tp*tn)/(fp*fn))
  i+=1

average_ROC(mean_fpr,tprs,aucs_ens,TP,TN,FP,FN)                             #plot average ROC
average_performance(aucs_ens,Accuracy,TP,TN,FP,FN)                          #print the average performance
print("Sensitivity (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(sn),np.std(sn)))
print("Specificity (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(sp),np.std(sp)))
print("Precision (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(pr),np.std(pr)))
print("FOR (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(FOR),np.std(FOR)))
print("DOR (Avg. +/- Std.) is  %0.3f +/- %0.3f" %(np.mean(DOR),np.std(DOR)))
